## Co-occurence Precip plot

In [ ]:
#Code for Figure 4 and Supplemental figure 1 and 2

In [ ]:
import numpy as np
import xarray as xr
import os 
import glob

In [ ]:
#Final so far
import dask
from dask.distributed import Client
import os

scheduler_file = os.path.join(os.environ["PSCRATCH"], "scheduler_jupyter.json")

dask.config.config["distributed"]["dashboard"]["link"] = "{JUPYTERHUB_SERVICE_PREFIX}proxy/{host}:{port}/status" 

client = Client(scheduler_file=scheduler_file)
client

In [ ]:
#Reading AR category
path = "/pscratch/sd/k/kquagra/Side_Jobs/Diya Collab/Categorised_ERA5_WCNA_data/"

# Match all years from 2001 to 2017
file_names = sorted(
    sum(
        [glob.glob(path + f"AR_categorization_{year}*.nc") for year in range(2001, 2018)],
        []
    )
)
#Reading AR Category files
ds = xr.open_mfdataset(file_names)
#print(ds)
AR_event_category = ds['AR_event_category']
print(AR_event_category)

In [ ]:
#Reading co occurence weather phenomena files
coocurrence_path = "/pscratch/sd/d/dkamnani/Results_data/training_label_*.nc"
ds_co = xr.open_mfdataset(coocurrence_path)
print(ds_co)
ds_co = ds_co['feat_comb_label']
#Making sure time steps match
matching_times = np.intersect1d(ds_co.time.values, AR_event_category.time.values)
ds_co = ds_co.sel(time=matching_times)
AR_event_category = AR_event_category.sel(time=matching_times)
print(ds_co)
print(AR_event_category)

In [ ]:
# Remap categories: 6 and 7 -> 5
AR_event_category = xr.where(AR_event_category.isin([6, 7]), 5, AR_event_category)

In [ ]:
#Matching array dimension
ds_co = ds_co.sel(latitude=slice(60, 0))
ds_co = ds_co.assign_coords(
    longitude=(((ds_co.longitude + 180) % 360) - 180)
)

# 2. Sort longitudes (important after reassignment to avoid disordered coords)
ds_co = ds_co.sortby('longitude')

# 3. Select the desired longitude range
ds_co = ds_co.sel(longitude=slice(-160, -50))
print(ds_co)

In [ ]:
#Teca data - Used for AR event identification
"""
teca_path = "/global/cfs/cdirs/m4374/catalogues/raw_catalogue_files/observations/teca_era5/ERA5_BARD_AR.2005*.nc4"
teca_data = xr.open_mfdataset(teca_path)
teca_data = teca_data["ar_binary_tag"]
print(teca_data)
"""
# Update path to include all years from 2001 to 2017
teca_path = "/global/cfs/cdirs/m4374/catalogues/raw_catalogue_files/observations/teca_era5/ERA5_BARD_AR.20*.nc4"

# Open all files and subset by time
teca_data = xr.open_mfdataset(teca_path, combine='by_coords')
teca_data = teca_data["ar_binary_tag"]

# Select years 2001 to 2017
teca_data = teca_data.sel(time=slice("2001-01-01", "2017-12-31"))

print(teca_data)

In [ ]:
time = ds_co.time.values
teca_data = teca_data.sel(time  = time)
print(teca_data)

In [ ]:
teca_data = teca_data.sel(latitude=slice(0, 60))
teca_data = teca_data.sortby('latitude', ascending = False)

teca_data = teca_data.assign_coords(longitude=(((teca_data.longitude + 180) % 360) - 180))
# 2. Sort longitudes (important after reassignment to avoid disordered coords)
teca_data = teca_data.sortby('longitude')

# 3. Select the desired longitude range
teca_data = teca_data.sel(longitude=slice(-160, -50))
print(teca_data)

In [ ]:
#precip
precip_path = "/pscratch/sd/d/dkamnani/Results_data/training_input_precip_*.nc"
ds_precip = xr.open_mfdataset(precip_path)
ds_precip = ds_precip['tp']
#Convert from meters to mm. 
ds_precip = ds_precip * 1000.0
ds_precip.attrs['units'] = 'mm'
matching_times = np.intersect1d(ds_precip.time.values, AR_event_category.time.values)
ds_precip = ds_precip.sel(time=matching_times)
#Matching array dimension
ds_precip = ds_precip.sel(latitude=slice(60, 0))
ds_precip = ds_precip.assign_coords(
    longitude=(((ds_precip.longitude + 180) % 360) - 180)
)

# 2. Sort longitudes (important after reassignment to avoid disordered coords)
ds_precip = ds_precip.sortby('longitude')

# 3. Select the desired longitude range
ds_precip = ds_precip.sel(longitude=slice(-160, -50))
print(ds_precip)

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
xr.align(ds_precip, teca_data, AR_event_category, ds_co, join="exact")


In [ ]:
tp_ar = ds_precip.where(teca_data == 1)

In [ ]:
categories = np.unique(AR_event_category.values)
categories = categories[categories > 0]   # drop 0 / no-AR

co_labels = np.unique(ds_co.values)
co_labels = co_labels[co_labels > 0]       # drop background


In [ ]:
# Mask everything to AR events only
cat_ar = AR_event_category.where(teca_data == 1)
feat_ar = ds_co.where(teca_data == 1)

In [ ]:
stacked = xr.Dataset(
    {
        "tp": tp_ar,
        "category": cat_ar,
        "cooccur": feat_ar,
    }
).stack(sample=("time", "latitude", "longitude"))


In [ ]:
stacked = stacked.dropna(dim="sample", subset=["tp", "category", "cooccur"])


In [ ]:
df = stacked.to_dataframe().reset_index(drop=True)

In [ ]:
# Convert to int (labels come in as floats after masking)
df["category"] = df["category"].astype(int)
df["cooccur"] = df["cooccur"].astype(int)

In [ ]:
print(df)

In [ ]:
# Drop rows where category == 0 or cooccur == 0
df = df[(df["category"] != 0) & (df["cooccur"] != 0)]


In [ ]:
import matplotlib.pyplot as plt
# Define the label index mapping
label_index = {
    1: 'AR',
    2: 'Front',
    3: 'MCS',
    4: 'LPS',
    5: 'AR + Front',
    6: 'AR + MCS',
    7: 'AR + LPS',
    8: 'Front + MCS',
    9: 'Front + LPS',
    10: 'MCS + LPS',
    11: 'AR + Front + MCS',
    12: 'AR + Front + LPS',
    13: 'AR + MCS + LPS',
    14: 'Front + MCS + LPS',
    15: 'All'
}

# Fractional contribution per category + cooccur
frac = df.groupby(["category", "cooccur"])["tp"].sum().unstack(fill_value=0)
frac = frac.div(frac.sum(axis=1), axis=0)
# Rename legend columns using label_index
frac = frac.rename(columns=label_index)

# Plot
plt.figure(figsize=(10,6))
ax = frac.plot(kind="bar")

plt.xlabel("Category")
plt.ylabel("Fraction of Total Precipitation")
plt.title("Fractional Precip Contribution by Category and Co-occurrence")

# Legend to the left
ax.legend(title="Co-occurrence", bbox_to_anchor=(-0.25, 1), loc="upper right")

plt.tight_layout()
plt.show()


## Seasonally

In [ ]:
import pandas as pd

# ---- Add season directly on stacked BEFORE dataframe conversion ----
def month_to_season(month):
    if month in [12, 1, 2]:
        return "DJF"
    elif month in [3, 4, 5]:
        return "MAM"
    elif month in [6, 7, 8]:
        return "JJA"
    else:
        return "SON"

# stacked must still contain time as a coordinate
stacked = stacked.assign_coords(
    season=("sample",
             pd.to_datetime(stacked["time"].values)
               .month
               .map(month_to_season))
)

# ---- Drop NaNs and convert to dataframe ----
stacked = stacked.dropna(dim="sample", subset=["tp", "category", "cooccur"])

df_season = stacked.to_dataframe().reset_index(drop=True)

# ---- Convert labels to int ----
df_season["category"] = df_season["category"].astype(int)
df_season["cooccur"]  = df_season["cooccur"].astype(int)


In [ ]:
# Drop rows where category == 0 or cooccur == 0
df_season = df_season[(df_season["category"] != 0) & (df_season["cooccur"] != 0)]


In [ ]:
import matplotlib.pyplot as plt

# Define the label index mapping
label_index = {
    1: 'AR',
    2: 'Front',
    3: 'MCS',
    4: 'LPS',
    5: 'AR + Front',
    6: 'AR + MCS',
    7: 'AR + LPS',
    8: 'Front + MCS',
    9: 'Front + LPS',
    10: 'MCS + LPS',
    11: 'AR + Front + MCS',
    12: 'AR + Front + LPS',
    13: 'AR + MCS + LPS',
    14: 'Front + MCS + LPS',
    15: 'All'
}

# Get all seasons in the data
seasons = df_season["season"].unique()

for season in seasons:
    # Subset for the season
    df_tmp = df_season[df_season["season"] == season]
    
    # Fractional contribution per category + cooccur
    frac = df_tmp.groupby(["category", "cooccur"])["tp"].sum().unstack(fill_value=0)
    frac = frac.div(frac.sum(axis=1), axis=0)
    
    # Rename co-occurrence columns
    frac = frac.rename(columns=label_index)
    
    # Plot
    plt.figure(figsize=(10,6))
    ax = frac.plot(kind="bar")
    
    plt.xlabel("Category")
    plt.ylabel("Fraction of Total Precipitation")
    plt.title(f"{season}: Fractional Precip Contribution by Category and Co-occurrence")
    
    # Legend to the left
    ax.legend(title="Co-occurrence", bbox_to_anchor=(-0.25, 1), loc="upper right")
    
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
# Prepare figure: 3 rows x 2 columns
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(14, 12))
axes = axes.flatten()
existing_labels = np.sort(df["cooccur"].dropna().unique())
# Discrete colormap with as many colors as unique co-occurrences
cmap15 = plt.get_cmap("tab20", len(existing_labels))

# BoundaryNorm maps category numbers to discrete colors
bounds = list(existing_labels) + [existing_labels.max() + 1]  # upper bound
norm15 = mcolors.BoundaryNorm(bounds, cmap15.N)


# map category → RGBA
category_colors = {k: cmap15(norm15(k + 0.5)) for k in existing_labels}
# --- Row 1: Annual ---
frac_all = df.groupby(["category","cooccur"])["tp"].sum().unstack(fill_value=0)
frac_all = frac_all.div(frac_all.sum(axis=1), axis=0)

frac_all.plot(kind="bar", ax=axes[0], legend=False,
              color=[category_colors[c] for c in frac_all.columns])
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Fraction of Total Precipitation")
axes[0].set_title("Annual")
axes[0].set_ylim(0, 0.75)

# --- Row 2: DJF | MAM ---
for i, season in enumerate(["DJF", "MAM"]):
    df_tmp = df_season[df_season["season"] == season]
    frac = df_tmp.groupby(["category","cooccur"])["tp"].sum().unstack(fill_value=0)
    frac = frac.div(frac.sum(axis=1), axis=0)
    
    ax = axes[2 + i]
    frac.plot(kind="bar", ax=ax, legend=False,
              color=[category_colors[c] for c in frac.columns])
    ax.set_xlabel("Category")
    ax.set_ylabel("Fraction of Total Precipitation")
    ax.set_title(season)
    ax.set_ylim(0, 0.75)

# --- Row 3: JJA | SON ---
for i, season in enumerate(["JJA", "SON"]):
    df_tmp = df_season[df_season["season"] == season]
    frac = df_tmp.groupby(["category","cooccur"])["tp"].sum().unstack(fill_value=0)
    frac = frac.div(frac.sum(axis=1), axis=0)
    
    ax = axes[4 + i]
    frac.plot(kind="bar", ax=ax, legend=False,
              color=[category_colors[c] for c in frac.columns])
    ax.set_xlabel("Category")
    ax.set_ylabel("Fraction of Total Precipitation")
    ax.set_title(season)
    ax.set_ylim(0, 0.75)

# --- Build legend content ---
present_co = set(frac_all.columns)

for season in ["DJF", "MAM", "JJA", "SON"]:
    df_tmp = df_season[df_season["season"] == season]
    frac_tmp = df_tmp.groupby(["category","cooccur"])["tp"].sum().unstack(fill_value=0)
    present_co |= set(frac_tmp.columns)

handles = [plt.Rectangle((0,0),1,1, color=category_colors[c]) 
           for c in sorted(present_co)]
labels = [label_index[c] for c in sorted(present_co)]

# --- Legend panel ---
axes[1].axis("off")
axes[1].legend(
    handles,
    labels,
    title="Co-occurrence",
    loc="center",
    fontsize=13,
    title_fontsize=15,
    ncol=1
)

plt.tight_layout()
plt.savefig("Supplementary_Figure1.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Figure 4 Final 
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
# --- Label mapping ---
label_index = {
    1: 'AR', 2: 'Front', 3: 'MCS', 4: 'LPS',
    5: 'AR + Front', 6: 'AR + MCS', 7: 'AR + LPS',
    8: 'Front + MCS', 9: 'Front + LPS',
    10: 'MCS + LPS', 11: 'AR + Front + MCS',
    12: 'AR + Front + LPS', 13: 'AR + MCS + LPS',
    14: 'Front + MCS + LPS', 15: 'All'
}
"""
# --- Colors ---
base_cmap = plt.get_cmap('tab20').colors
category_colors = {k: base_cmap[k-1] for k in label_index.keys()}
"""
existing_labels = np.sort(df["cooccur"].dropna().unique())
# Discrete colormap with as many colors as unique co-occurrences
cmap15 = plt.get_cmap("tab20", len(existing_labels))

# BoundaryNorm maps category numbers to discrete colors
bounds = list(existing_labels) + [existing_labels.max() + 1]  # upper bound
norm15 = mcolors.BoundaryNorm(bounds, cmap15.N)


# map category → RGBA
category_colors = {k: cmap15(norm15(k + 0.5)) for k in existing_labels}
# --- Compute 90th percentile threshold ---
#p90 = df["tp"].quantile(0.90)
# --- Compute 90th percentile threshold (exclude zero precipitation) ---
p90 = df.loc[df["tp"] > 0, "tp"].quantile(0.90)

# --- Extreme precipitation subset ---
df_ext = df[df["tp"] >= p90]
df_season_ext = df_season[df_season["tp"] >= p90]

# --- Prepare figure ---
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(16, 12))

# ==========================================================
# Row 1 — Annual totals (LEFT) + Annual extremes (RIGHT)
# ==========================================================

# --- Annual total ---
frac_all = df.groupby(["category","cooccur"])["tp"].sum().unstack(fill_value=0)
frac_all = frac_all.div(frac_all.sum(axis=1), axis=0)
colors = [category_colors.get(c, "gray") for c in frac_all.columns]

frac_all.plot(kind="bar", ax=axes[0,0], color=colors, legend=False)
axes[0,0].set_title("Annual (All Precip)")
axes[0,0].set_ylabel("Fraction of Total Precip")

# --- Annual extremes ---
frac_all_ext = df_ext.groupby(["category","cooccur"])["tp"].sum().unstack(fill_value=0)
frac_all_ext = frac_all_ext.div(frac_all_ext.sum(axis=1), axis=0)
colors = [category_colors.get(c, "gray") for c in frac_all_ext.columns]

frac_all_ext.plot(kind="bar", ax=axes[0,1], color=colors, legend=False)
axes[0,1].set_title("Annual Extremes (≥90th %)")

# ==========================================================
# Row 2 — DJF totals + DJF extremes
# ==========================================================

for col, data, title in zip(
    [0,1],
    [df_season, df_season_ext],
    ["DJF", "DJF Extremes (≥90th %)"]
):
    tmp = data[data["season"]=="DJF"]
    frac = tmp.groupby(["category","cooccur"])["tp"].sum().unstack(fill_value=0)
    frac = frac.div(frac.sum(axis=1), axis=0)

    colors = [category_colors.get(c, "gray") for c in frac.columns]
    frac.plot(kind="bar", ax=axes[1,col], color=colors, legend=False)

    axes[1,col].set_title(title)
    axes[1,col].set_ylabel("Fraction of Total Precip")

# ==========================================================
# Row 3 — JJA totals + JJA extremes
# ==========================================================

for col, data, title in zip(
    [0,1],
    [df_season, df_season_ext],
    ["JJA", "JJA Extremes (≥90th %)"]
):
    tmp = data[data["season"]=="JJA"]
    frac = tmp.groupby(["category","cooccur"])["tp"].sum().unstack(fill_value=0)
    frac = frac.div(frac.sum(axis=1), axis=0)

    colors = [category_colors.get(c, "gray") for c in frac.columns]
    frac.plot(kind="bar", ax=axes[2,col], color=colors, legend=False)

    axes[2,col].set_title(title)
    axes[2,col].set_ylabel("Fraction of Total Precip per Category")

# ==========================================================
# Legend (single, shared)
# ==========================================================

present_co = set(frac_all.columns) | set(frac_all_ext.columns)

handles = [plt.Rectangle((0,0),1,1, color=category_colors[c]) 
           for c in sorted(present_co)]
labels = [label_index[c] for c in sorted(present_co)]

fig.legend(
    handles,
    labels,
    title="Co-occurrence",
    loc="lower center",
    fontsize=13,
    title_fontsize=15,
    ncol=4,
    bbox_to_anchor=(0.5, -0.03)
)
for ax in axes.flat:
    ax.set_ylim(0, 0.7)
plt.tight_layout(rect=[0,0.05,1,1])
plt.savefig("Precip_by_cooccurrence_extremes.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Supplemental Figure #2 Final 
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors

# --- Label mapping ---
label_index = {
    1: 'AR', 2: 'Front', 3: 'MCS', 4: 'LPS',
    5: 'AR + Front', 6: 'AR + MCS', 7: 'AR + LPS',
    8: 'Front + MCS', 9: 'Front + LPS',
    10: 'MCS + LPS', 11: 'AR + Front + MCS',
    12: 'AR + Front + LPS', 13: 'AR + MCS + LPS',
    14: 'Front + MCS + LPS', 15: 'All'
}

# --- Colormap ---
existing_labels = np.sort(df["cooccur"].dropna().unique())
cmap15 = plt.get_cmap("tab20", len(existing_labels))
bounds = list(existing_labels) + [existing_labels.max() + 1]
norm15 = mcolors.BoundaryNorm(bounds, cmap15.N)

category_colors = {k: cmap15(norm15(k + 0.5)) for k in existing_labels}

# --- Extreme threshold ---
#p90 = df["tp"].quantile(0.90)

# --- Compute 90th percentile threshold (exclude zero precipitation) ---
p90 = df.loc[df["tp"] > 0, "tp"].quantile(0.90)

df_ext = df[df["tp"] >= p90]
df_season_ext = df_season[df_season["tp"] >= p90]

# --- Helper function ---
def plot_fraction(data, ax, title):
    frac = data.groupby(["category","cooccur"])["tp"].sum().unstack(fill_value=0)
    frac = frac.div(frac.sum(axis=1), axis=0)

    # enforce consistent column order
    frac = frac.reindex(columns=sorted(existing_labels), fill_value=0)

    colors = [category_colors.get(c, "gray") for c in frac.columns]
    frac.plot(kind="bar", ax=ax, color=colors, legend=False)

    ax.set_title(title)
    ax.set_ylabel("Fraction")
    ax.set_ylim(0, 1)

# --- Figure ---
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# ==========================================================
# Row 1
# ==========================================================
plot_fraction(df_ext, axes[0,0], "Annual Extremes (≥90th %)")

# --- Legend panel (row 1 col 2) ---
axes[0,1].axis("off")

handles = [plt.Rectangle((0,0),1,1, color=category_colors[c]) 
           for c in sorted(existing_labels)]
labels = [label_index[c] for c in sorted(existing_labels)]

axes[0,1].legend(
    handles,
    labels,
    title="Co-occurrence",
    loc="center",
    fontsize=12,
    title_fontsize=14,
    ncol=2
)

# ==========================================================
# Row 2
# ==========================================================
plot_fraction(
    df_season_ext[df_season_ext["season"]=="DJF"],
    axes[1,0],
    "DJF Extremes"
)

plot_fraction(
    df_season_ext[df_season_ext["season"]=="MAM"],
    axes[1,1],
    "MAM Extremes"
)

# ==========================================================
# Row 3
# ==========================================================
plot_fraction(
    df_season_ext[df_season_ext["season"]=="JJA"],
    axes[2,0],
    "JJA Extremes"
)

plot_fraction(
    df_season_ext[df_season_ext["season"]=="SON"],
    axes[2,1],
    "SON Extremes"
)
for ax in axes.flat:
    ax.set_ylim(0, 0.7)

# --- Layout ---
plt.tight_layout()
plt.savefig("Supplementary_Figure2.pdf", dpi=300, bbox_inches="tight")
plt.show()